In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

IMPORTS & SETUP

In [ ]:
# CELL 1: Import all libraries
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
from tensorflow.keras.preprocessing import image_dataset_from_directory
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print("✅ Libraries imported successfully!")
print("TensorFlow version:", tf.__version__)

CELL 2: DATA GENERATORS

In [ ]:
# CELL 2: Setup data generators with correct preprocessing
print("📦 Setting up data generators...")

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,  # MobileNetV2 specific preprocessing
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    validation_split=0.2
)

# UPDATE THIS PATH - మీ dataset path ఇవ్వండి
dataset_path = "/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color"

print("✅ Data generators ready")

CELL 3: LOAD DATA

In [ ]:
# CELL 3: Load training and validation data
print("⏳ Loading data...")

train_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="training"
)

val_generator = datagen.flow_from_directory(
    dataset_path,
    target_size=(224, 224),
    batch_size=32,
    class_mode="categorical",
    subset="validation"
)

num_classes = train_generator.num_classes
class_names = list(train_generator.class_indices.keys())

print(f"\n✅ Found {num_classes} classes")
print(f"✅ Training samples: {train_generator.samples}")
print(f"✅ Validation samples: {val_generator.samples}")
print(f"\n📋 First 5 classes: {class_names[:5]}")

CELL 4: BUILD MODEL

In [ ]:
# CELL 4: Build improved model architecture
print("🏗️ Building model...")

base_model = MobileNetV2(
    weights="imagenet",
    include_top=False,
    input_shape=(224, 224, 3)
)
base_model.trainable = False  # Freeze for phase 1

# Improved architecture with BatchNormalization
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(256, activation="relu")(x)
x = BatchNormalization()(x)
x = Dropout(0.4)(x)
x = Dense(128, activation="relu")(x)
x = Dropout(0.3)(x)
output = Dense(num_classes, activation="softmax")(x)

model = Model(inputs=base_model.input, outputs=output)

model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

print("✅ Model built successfully!")
model.summary()

 CELL 5: CALLBACKS

In [ ]:
# CELL 5: Setup callbacks
print("🎯 Setting up callbacks...")

checkpoint = ModelCheckpoint(
    'best_model.keras',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.3,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

callbacks = [checkpoint, early_stop, reduce_lr]
print("✅ Callbacks ready")

CELL 6: PHASE 1 TRAINING

In [ ]:
# CELL 6: Phase 1 training (frozen base)
print("\n" + "="*60)
print("🚀 PHASE 1: TRAINING STARTED (10 epochs)")
print("="*60)
print("⏱️ This will take 45-60 minutes...")

history1 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=10,
    callbacks=callbacks,
    verbose=1
)

print("✅ Phase 1 complete!")

PHASE 2 FINE-TUNING

In [ ]:
# CELL 7: Phase 2 fine-tuning
print("\n" + "="*60)
print("🚀 PHASE 2: FINE-TUNING STARTED (5 epochs)")
print("="*60)  # ← ఇది సరైనది (""*60" not "=60")
print("⏱️ This will take 20-30 minutes...")

# Unfreeze base model
base_model.trainable = True

# Freeze first 100 layers (preserves features)
for layer in base_model.layers[:100]:
    layer.trainable = False

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=5,
    callbacks=callbacks,
    verbose=1
)

print("✅ Phase 2 complete!")
print("🎉 TOTAL TRAINING COMPLETE!")

CELL 8: SAVE MODEL & CLASS NAMES

In [ ]:
# CELL 8: Save final model and class names
print("\n" + "="*60)
print("💾 SAVING FINAL MODEL")
print("="*60)

from tensorflow.keras.models import load_model
import pickle
import os

# ============================================
# 1. SAVE THE MODEL
# ============================================
model_filename = 'plant_disease_final_model.keras'
model.save(model_filename)
print(f"✅ Model saved as: {model_filename}")

# Also save the best model from checkpoint
if os.path.exists('best_model.keras'):
    best_model = load_model('best_model.keras')
    best_model.save('plant_disease_best_model.keras')
    print("✅ Best model saved as: plant_disease_best_model.keras")

# ============================================
# 2. SAVE CLASS NAMES
# ============================================
with open('class_names.pkl', 'wb') as f:
    pickle.dump(class_names, f)
print("✅ Class names saved as: class_names.pkl")

# ============================================
# 3. VERIFY FILES
# ============================================
print("\n📁 Verifying saved files:")
for file in ['plant_disease_final_model.keras', 'class_names.pkl']:
    if os.path.exists(file):
        size = os.path.getsize(file)
        if size > 1024*1024:
            print(f"   ✅ {file} - {size/(1024*1024):.2f} MB")
        else:
            print(f"   ✅ {file} - {size/1024:.2f} KB")

print("\n" + "="*60)
print("✅ MODEL SAVED SUCCESSFULLY!")
print("="*60)

CELL 9: FINAL PREDICTION FUNCTION with Leaf Name, Disease & Stress Level

In [ ]:
# CELL 9: FINAL PREDICTION FUNCTION - Leaf Name, Disease & Stress Level
print("\n" + "="*60)
print("🔍 CREATING FINAL PREDICTION FUNCTION")
print("="*60)

from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os

# ============================================
# LOAD THE SAVED MODEL
# ============================================
print("\n📁 Loading saved model...")

if os.path.exists('plant_disease_final_model.keras'):
    model = load_model('plant_disease_final_model.keras')
    print("✅ Model loaded: plant_disease_final_model.keras")
else:
    model = model  # Use the current model from training
    print("✅ Using current training model")

# Load class names
with open('class_names.pkl', 'rb') as f:
    class_names = pickle.load(f)
print(f"✅ Class names loaded: {len(class_names)} classes")

# ============================================
# FUNCTION TO EXTRACT LEAF NAME
# ============================================
def extract_leaf_name(disease_name):
    """
    Extract leaf name from disease string
    Example: "Apple__Apple_scab" -> "Apple"
    """
    parts = disease_name.split('__')
    if len(parts) > 0:
        return parts[0].replace('_', ' ')
    return "Unknown"

# ============================================
# STRESS LEVEL FUNCTION
# ============================================
def get_stress_level(disease_name, confidence):
    """
    Determine stress level based on disease and confidence
    """
    disease_lower = disease_name.lower()
    
    # Extract just the disease part for better matching
    disease_part = disease_lower.split('__')[-1] if '__' in disease_lower else disease_lower
    
    # Healthy case
    if 'healthy' in disease_lower:
        return {
            'level': '🟢 NO STRESS',
            'color': 'green',
            'emoji': '✅',
            'action': 'No action needed - Plant is healthy',
            'severity': 0
        }
    
    # Define disease severity categories
    severe_diseases = ['scab', 'blight', 'rust', 'mildew', 'rot', 'canker', 'wilt', 'late_blight']
    moderate_diseases = ['spot', 'mosaic', 'curl', 'virus', 'bacterial', 'early_blight']
    mild_diseases = ['leaf', 'yellow', 'streak']
    
    # Check severity
    if any(d in disease_lower for d in severe_diseases):
        base_severity = 3
        category = "SEVERE"
        emoji = "🔴"
        action = "⚠️ URGENT: Immediate treatment required! Apply fungicide/pesticide."
    elif any(d in disease_lower for d in moderate_diseases):
        base_severity = 2
        category = "MODERATE"
        emoji = "🟠"
        action = "⚕️ Start treatment within 2-3 days. Remove affected leaves."
    elif any(d in disease_lower for d in mild_diseases):
        base_severity = 1
        category = "MILD"
        emoji = "🟡"
        action = "🔍 Monitor plant daily. Consider organic treatment."
    else:
        base_severity = 1
        category = "MILD"
        emoji = "🟡"
        action = "🔍 Monitor closely. Consult if symptoms worsen."
    
    # Adjust based on confidence
    if confidence > 0.9:
        stress_level = f"{emoji} HIGH {category} STRESS"
        if base_severity == 3:
            action = "⚠️ CRITICAL: Immediate professional consultation required!"
    elif confidence > 0.7:
        stress_level = f"{emoji} MODERATE {category} STRESS"
    else:
        stress_level = f"{emoji} LOW {category} STRESS"
    
    return {
        'level': stress_level,
        'action': action,
        'emoji': emoji
    }

# ============================================
# MAIN PREDICTION FUNCTION
# ============================================
def predict_leaf_disease(img_path, show_plot=True):
    """
    Predict leaf name, disease, and stress level from image
    Returns: (leaf_name, disease_name, confidence, stress_level, action)
    """
    # Load and preprocess image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    
    # Predict
    predictions = model.predict(img_array, verbose=0)[0]
    
    # Get top prediction
    top_idx = np.argmax(predictions)
    disease_name = class_names[top_idx]
    confidence = predictions[top_idx]
    
    # Extract leaf name
    leaf_name = extract_leaf_name(disease_name)
    
    # Get stress level
    stress_info = get_stress_level(disease_name, confidence)
    
    if show_plot:
        plt.figure(figsize=(14, 5))
        
        # Input image
        plt.subplot(1, 3, 1)
        plt.imshow(img)
        plt.axis('off')
        plt.title('🌿 Input Leaf', fontsize=12, fontweight='bold')
        
        # Top 3 predictions
        plt.subplot(1, 3, 2)
        top_3_idx = np.argsort(predictions)[-3:][::-1]
        top_3_names = [class_names[i].split('__')[-1].replace('_', ' ') for i in top_3_idx]
        top_3_conf = [predictions[i] for i in top_3_idx]
        
        colors = ['#ff6b6b', '#4ecdc4', '#45b7d1']
        bars = plt.barh(range(3), top_3_conf, color=colors)
        plt.yticks(range(3), [f"{n[:20]}..." if len(n)>20 else n for n in top_3_names])
        plt.xlabel('Confidence')
        plt.title('📊 Top 3 Diseases', fontsize=12, fontweight='bold')
        plt.xlim(0, 1)
        
        for i, (bar, conf) in enumerate(zip(bars, top_3_conf)):
            plt.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                    f'{conf:.1%}', va='center')
        
        # Stress level box
        plt.subplot(1, 3, 3)
        plt.axis('off')
        
        # Color mapping
        color_map = {
            '🔴': '#ffcccc',
            '🟠': '#ffe6cc',
            '🟡': '#ffffcc',
            '🟢': '#ccffcc'
        }
        box_color = color_map.get(stress_info['emoji'], '#f0f0f0')
        
        result_text = f"""
        {stress_info['emoji']} FINAL DIAGNOSIS {stress_info['emoji']}
        {'═'*30}
        
        🌱 LEAF: {leaf_name}
        
        🦠 DISEASE: {disease_name.split('__')[-1].replace('_', ' ')}
        
        📊 CONFIDENCE: {confidence:.1%}
        
        {stress_info['level']}
        
        💊 ACTION:
        {stress_info['action']}
        {'═'*30}
        """
        
        plt.text(0.1, 0.5, result_text, fontsize=10, 
                verticalalignment='center',
                transform=plt.gca().transAxes,
                bbox=dict(boxstyle='round', facecolor=box_color, 
                         alpha=0.9, pad=1))
        
        plt.tight_layout()
        plt.show()
    
    # Print results
    print("\n" + "="*60)
    print("🎯 FINAL PREDICTION RESULTS")
    print("="*60)
    print(f"\n🌱 LEAF NAME: {leaf_name}")
    print(f"🦠 DISEASE: {disease_name.split('__')[-1].replace('_', ' ')}")
    print(f"📊 CONFIDENCE: {confidence:.2%}")
    print(f"{stress_info['level']}")
    print(f"\n💊 RECOMMENDED ACTION:")
    print(f"   {stress_info['action']}")
    print("\n" + "="*60)
    
    return {
        'leaf_name': leaf_name,
        'disease': disease_name,
        'confidence': confidence,
        'stress_level': stress_info['level'],
        'action': stress_info['action']
    }

print("\n✅ Prediction function created successfully!")
print("\n📝 USAGE:")
print("   result = predict_leaf_disease('image_path.jpg')")
print("   print(result['leaf_name'])     # Shows leaf name")
print("   print(result['disease'])       # Shows disease")
print("   print(result['stress_level'])  # Shows stress level")

CELL 10: TEST WITH YOUR IMAGE

In [ ]:
# CELL 10: Test with your image
print("\n" + "="*60)
print("📸 TESTING WITH YOUR IMAGE")
print("="*60)

# Update this path to your test image
test_image_path = "/kaggle/input/datasets/ganuboyinanikhitha/testing-images/strawberry.jpg"

if os.path.exists(test_image_path):
    print(f"\n🔍 Testing: {test_image_path}")
    result = predict_leaf_disease(test_image_path, show_plot=True)
    
    # Print summary
    print("\n📋 QUICK SUMMARY:")
    print(f"   🌱 Leaf: {result['leaf_name']}")
    print(f"   🦠 Disease: {result['disease'].split('__')[-1].replace('_', ' ')}")
    print(f"   📊 Confidence: {result['confidence']:.1%}")
    print(f"   {result['stress_level']}")
else:
    print(f"\n⚠️ Image not found: {test_image_path}")
    print("Please update the path to your test image")
    
    # List available files in directory
    base_dir = "/kaggle/input/"
    if os.path.exists(base_dir):
        print("\n📁 Available input directories:")
        for d in os.listdir(base_dir)[:5]:
            print(f"   - {base_dir}{d}")

CELL 11: DOWNLOAD FILES

In [ ]:
# CELL 11: Download model and class names
print("\n" + "="*60)
print("📥 DOWNLOAD FILES")
print("="*60)

try:
    from IPython.display import FileLink, display
    
    print("\n📎 Click to download model:")
    display(FileLink('plant_disease_final_model.keras'))
    
    print("\n📎 Click to download class names:")
    display(FileLink('class_names.pkl'))
    
    print("\n💡 Save these files on your laptop for future use!")
    
except:
    print("\nTo download files manually:")
    print("1. Go to 'Data' tab on right side")
    print("2. Find files in 'Output' section")
    print("3. Click on file → Download")